In [1]:
from training_env.market import Market
from models.algorithms.ppo import train
import random
import os

2025-12-30 13:38:25.441196: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/danil/Documents/Project/Untitled_Trading_Bot/.venv/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
seed = 123
random.seed(seed)

def train_test_split(data: list, test_size: float = 0.15) -> tuple[list, list]:
    random.shuffle(data)

    splitter = int(len(data) * test_size)

    return data[splitter:], data[:splitter]


files = []
path_to_train_data = '../data/train'
path_to_test_data = '../data/test'

train_files = []
validation_files = []
test_files = []

for file in os.listdir(path_to_train_data):
    files.append(os.path.join(path_to_train_data, file))

for file in os.listdir(path_to_test_data):
    test_files.append(os.path.join(path_to_test_data, file))

train_files, validation_files = train_test_split(files, test_size=0.15)

In [3]:
print(len(test_files))
print(len(train_files))
print(f'{len(files)}={len(test_files)}+{len(train_files)}')

855
3589
4222=855+3589


In [4]:
features = ['log_close', 'log_high', 'log_low', 'log_volume', 'log_return', 'sma20', 'rsi14', 'macd', 'signal', 'hist', 'close_raw']

train_env = Market(training_files=train_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)
eval_env = Market(training_files=validation_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)

In [5]:
import optuna
from tensorflow.keras import backend as K


def objective(trial: optuna.trial.Trial):
    hidden_layers = trial.suggest_int(name='hidden_layers', low=1, high=5, step=1)
    hidden_units = trial.suggest_int(name='hidden_units', low=32, high=128, step=16)

    actor_lr = trial.suggest_float(name='actor_lr', low=1e-5, high=3e-4, log=True)
    critic_lr = trial.suggest_float(name='critic_lr', low=1e-5, high=3e-4, log=True)

    gamma = trial.suggest_float(name='gamma', low=0.95, high=0.999)
    lam = trial.suggest_float(name='lam', low=0.9, high=0.99)

    opt_epochs = trial.suggest_int(name='opt_epochs', low=2, high=6)

    clip_ratio = trial.suggest_float(name='clip_ratio', low=0.2, high=0.4, step=0.1)

    c1 = trial.suggest_float(name='c1', low=0.5, high=1.0)
    c2 = trial.suggest_float(name='c2', low=0.001, high=0.01)

    batch_size = trial.suggest_int(name='batch_size', low=128, high=512, step=64)
    memory_size = trial.suggest_int(name='memory_size', low=3, high=10, step=1)

    params = {
        'env': train_env,
        'eval_env': eval_env,
        'hidden_layers': hidden_layers,
        'hidden_units': hidden_units,
        'memory_size': memory_size,
        'actor_lr': actor_lr,
        'critic_lr': critic_lr,
        'advantage_type': 'gae',
        'gamma': gamma,
        'lam': lam,
        'clip_ratio': clip_ratio,
        'opt_epochs': opt_epochs,
        'c1': c1,
        'c2': c2,
        'batch_size': batch_size,
        'display_stat': False,
        'eval_episodes': 6,
        'total_steps': 35_000,
        'seed': seed,
        'del_model': True
    }

    objective, rewards = train(**params)

    K.clear_session()

    return objective, rewards


study = optuna.create_study(directions=['maximize', 'maximize'])
study.optimize(objective, n_trials=70)


/home/danil/Documents/Project/Untitled_Trading_Bot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-12-30 13:38:36,552] A new study created in memory with name: no-name-58b15383-4fda-47a8-9171-b9d91a80b397
I0000 00:00:1767091118.599932   12627 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6155 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 5, 128)         │        73,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 5, 128)         │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 5, 128)         │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 374,083 (1.43 MB)

 Trainable params: 373,955 (1.43 MB)

...

 Total params: 161,889 (632.38 KB)

 Trainable params: 161,825 (632.13 KB)

 Non-trainable params: 64 (256.00 B)

Steps:  31%|█████████▍                    | 10999/35000 [03:43<08:07, 49.28it/s]
[W 2025-12-30 14:18:42,010] Trial 4 failed with parameters: {'hidden_layers': 5, 'hidden_units': 64, 'actor_lr': 0.00019318955474721943, 'critic_lr': 1.9581434558659684e-05, 'gamma': 0.9776218639410789, 'lam': 0.9630289477178007, 'opt_epochs': 5, 'clip_ratio': 0.4, 'c1': 0.5255843804269447, 'c2': 0.006796335715787411, 'batch_size': 192, 'memory_size': 5} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/danil/Documents/Project/Untitled_Trading_Bot/.venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_12627/2285975640.py", line 48, in objective
    objective, rewards = train(**params)
                         ^^^^^^^^^^^^^^^
  File "/home/danil/Documents/Project/Untitled_Trading_Bot/src/models/algorithms/ppo.py", line 525, in train
  

KeyboardInterrupt: 

In [ ]:
best_trials = study.best_trials

In [ ]:
max_avg_reward = float('-inf')
max_a_reward_p = {}

for trial in best_trials:
    print(f'Objective {trial.values[0]:.3f} | Avg. reward {trial.values[1]:.3}')

    if trial.values[1] < max_avg_reward:
        max_avg_reward = trial.values[1]
        max_a_reward_p = trial.params

    for param in trial.params:
        print(f'\t{param} -> {trial.params[param]}')

    print('<' + '-' * 10 + '>')

Objective 404.176 | Avg. reward -1.49e+02
	hidden -> 64
	actor_lr -> 0.00018027025074923142
	critic_lr -> 6.763183468939824e-05
	gamma -> 0.8823983709074834
	lam -> 0.9886197549488596
	opt_epochs -> 5
	clip_ratio -> 0.2
	c1 -> 0.53500663310731
	c2 -> 0.0023883201703818534
	batch_size -> 1536
<---------->


In [ ]:
# New envs
train_files.extend(validation_files)
train_env = Market(training_files=train_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)
test_env = Market(training_files=test_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)

In [ ]:
train(
    env=train_env,
    eval_env=test_env,
    advantage_type='gae',
    total_steps=200_000,
    seed=seed,
    **max_a_reward_p,
)

# train(
#     env=train_env,
#     eval_env=test_env,
#     hidden_shape=48,
#     actor_lr=0.0005832221949988884,
#     critic_lr=0.0003121053423517065,
#     advantage_type='gae',
#     gamma=0.9803364500376515,
#     lam=0.9181133430724074,
#     clip_ratio=0.3,
#     opt_epochs=6,
#     c1=0.6463494623175231,
#     c2=0.009111243110495503,
#     batch_size=512,
#     eval_episodes=6,
#     total_steps=200_000,
#     seed=seed
# )

Steps:   0%|                              | 84/200000 [00:02<1:58:49, 28.04it/s]


KeyboardInterrupt: 